# Example for line density calculation
* based on geo-referenced building geometries and line geometries (streets)

## Imports and parameter definition

In [ ]:
import os
import numpy as np
import geoppi
import geopandas as gp
import pandas as pd
from pathlib import Path

# Define cooridnate system
cs = 'EPSG:25832'

## Load example data
flp = os.getcwd() / Path(r'data/exampleNetwork2/')

# Define whether a randomized sampling in each hexagon with aim connection ratio shall take place
rand_sampling = True

# Number of randomized samples for selection of buildings to attain aimAG (sample with median heat demand is chosen)
nSamples = 1

# Define minimum number of buildings which shall be found within single hexagon below which the aimAG is ignored and ALL buildings are considered for calculation
nBuildings_min = 1

# Define aim of connection ratio within each subdivision of the regarded area in hex_div
target_CR = 1

# Define hexagon-specific connection ratio as attribute name of its value in hex
attr_hex_CR = 'connectionRatio'

# Define attribute from buildings for calculation of line density
target_attr = 'demand_use_th'

# Define additional attributes from the buildings layer which shall be summed on closest line objects
summed_target_attrs = ['demand_use_th_2045_san']

# Define maximum distance between buildings and line objects to include them into calculation of line density
# -> Example uses function-like distance calculation
def calc_distance_from_heat_demand(
    heat_demand, 
    min_line_density = 1000, # At least line density of 1 MWh/m at house connection line to assign closest line to polygon
    minLimit = 10,
    maxLimit = 100):

    return(min(maxLimit, max(minLimit, heat_demand/min_line_density)) )

## Example 1)

### Loading data

In [ ]:
lines = gp.read_file(flp / Path(r'streets.gpkg')).to_crs(cs)
lines["unique_ID_lines"] = np.arange(len(lines)).astype(int)

buildings = gp.read_file(flp / Path(r'buildings.gpkg')).to_crs(cs)
buildings["unique_ID_polys"] = np.arange(len(buildings)).astype(int)

hex = None#gp.read_file(flp / Path(r'hex_div.gpkg')).to_crs(cs)

In [ ]:
### Start calculation
lines_out, polys_out = geoppi.sum_attributes_on_lines(
    polygons = buildings,
    lines = lines,
    spatial_distribution = hex,
    rand_sampling = rand_sampling,
    nSamples = nSamples,
    nPolygons_min = nBuildings_min,
    target_connection_ratio = target_CR,
    spatial_connection_ratio = attr_hex_CR,
    target_attr = 'demand_use_th',
    agg_func = 'median',
    additional_attr = summed_target_attrs,
    polygons_uniqueID = "unique_ID_polys",
    lines_uniqueID = "unique_ID_lines",
    func_max_distance = None #calc_distance_from_heat_demand
)

lines_out[f'ld_{target_attr}_MWh_per_m'] = lines_out[f'summed_{target_attr}'] / 1e03 / lines_out.geometry.length

### Plotting

In [ ]:
from config_plotting import *

fig, ax = plt.subplots(constrained_layout = True)
lines_out.plot(ax = ax,
column = f'ld_{target_attr}_MWh_per_m', 
    cmap=plt.get_cmap('viridis'),
    vmin = 0,
    vmax = 5,
    categorical = False,
    missing_kwds = dict(color = 'grey', label = '-'),
    legend = True)

cbar_ax = fig.axes[-1]
cbar_ax.set_ylabel("Line density (MWh/m)")

## Example 2)
* Pre-processing of matching polygons->lines by additional attributes:
    * determination of adjacent streets to buildings
    * determination of matching addresses/street names


### Loading data

In [ ]:
lines = gp.read_file(flp / Path(r'streets.gpkg')).to_crs(cs)
lines["unique_ID_lines"] = np.arange(len(lines)).astype(int)

buildings = gp.read_file(flp / Path(r'buildings.gpkg')).to_crs(cs)
buildings["unique_ID_polys"] = np.arange(len(buildings)).astype(int)

parcels = gp.read_file(flp / Path(r"parcels.gpkg")).to_crs(cs)

## Pre-processing of parcel data: Delet all parcels with traffic-related function
# Such parcels are identified by an intersection with street layer
temp_parcel = parcels.sjoin(lines, predicate = "intersects", how = "left")
temp_parcel = temp_parcel[~temp_parcel.index.duplicated()]

parcels = parcels[temp_parcel["index_right"].isna().values].reset_index(drop = True)

In [ ]:
## Match buildings to parcels
buildings = geoppi.assign_attr_by_max_intersection_area(gp1 = buildings, gp_source = parcels, gp1_id = "unique_ID_polys", attr = ["idflurst"]).rename(columns = {"idflurst":"parc_id"})

## Determine adjacent lines to parcels
parcels_out, connLines =  geoppi.get_adjacent_lines(
    polygons = parcels,
    unique_ID_polys = "idflurst",
    unique_ID_lines = "unique_ID_lines",
    lines = lines,
    return_connection_lines = True,
    blocking_polygons = parcels,
    resolution = 5,
    dist = 50,
    nLines = 5
)

# Transfer adjacent line IDs to buildings
buildings_out = pd.merge(buildings, parcels_out[["idflurst", "adjacentLine_IDs", "adjacentDists"]], left_on = "parc_id", right_on = "idflurst", how = "left").drop(columns = ["idflurst"])


In [ ]:
## Create dictionary for matching buildings to lines in advance
# Use closest adjacent line to matched parcel for each building
# Dictionary provided to function sum_attributes_on_lines can contain arbitrary matching (see uncommented example dictionary in function call)

dict_buildID_lineID = dict(zip(list(buildings_out["unique_ID_polys"]), [n[0] if (isinstance(n, list) and len(n)>0) else n if (n is not None and not isinstance(n, list)) else None for n in buildings_out["adjacentLine_IDs"]]))


In [ ]:
### Start calculation of line density
lines_out, polys_out = geoppi.sum_attributes_on_lines(
    polygons = buildings_out,
    lines = lines,
    spatial_distribution = hex,
    rand_sampling = rand_sampling,
    nSamples = nSamples,
    nPolygons_min = nBuildings_min,
    target_connection_ratio = target_CR,
    spatial_connection_ratio = attr_hex_CR,
    target_attr = 'demand_use_th',
    agg_func = 'median',
    additional_attr = summed_target_attrs,
    polygons_uniqueID = "unique_ID_polys",
    lines_uniqueID = "unique_ID_lines",
    dict_polyID_lineID = dict_buildID_lineID,,#{0:50, 1:50, 100:50,101:50},
    func_max_distance = None #calc_distance_from_heat_demand
)

lines_out[f'ld_{target_attr}_MWh_per_m'] = lines_out[f'summed_{target_attr}'] / 1e03 / lines_out.geometry.length

### Plotting

In [ ]:
from config_plotting import *

fig, ax = plt.subplots(constrained_layout = True)
lines_out.plot(ax = ax,
column = f'ld_{target_attr}_MWh_per_m', 
    cmap=plt.get_cmap('viridis'),
    vmin = 0,
    vmax = 5,
    categorical = False,
    missing_kwds = dict(color = 'grey', label = '-'),
    legend = True)

cbar_ax = fig.axes[-1]
cbar_ax.set_ylabel("Line density (MWh/m)")